# Harmoniq — Notebook proof (PRIORITIES commit 1)

**Chain:** ingest → ffmpeg (44.1 kHz mono) → Demucs `htdemucs_6s` → Librosa summary → Basic Pitch → `.gp5`

**Expectations**
- **CPU:** first Demucs run downloads weights; separation is slow without GPU.
- **GPU:** install a CUDA-enabled `torch`/`torchaudio` build before `demucs` for large speedups (see `backend/README.md`).
- **Basic Pitch:** `pip install -e ".[basicpitch]"` from `backend/`; on Windows/Linux + Python 3.11+ this may fail — use macOS or a conda env documented in the Spotify/basic-pitch repo.

Run this notebook from the **`backend/`** directory (or set `BACKEND_ROOT` below to the folder that contains `app/`).

In [4]:
from __future__ import annotations

import shutil
from pathlib import Path

# Project root = parent of this notebook's folder (backend/)
BACKEND_ROOT = Path.cwd()
if not (BACKEND_ROOT / "app" / "pipeline_proof.py").is_file():
    BACKEND_ROOT = Path("..").resolve()

DATA = BACKEND_ROOT / "data" / "research_notebook"
DATA.mkdir(parents=True, exist_ok=True)

# --- Configure one of these ---
LOCAL_AUDIO: Path | None = None  # e.g. Path(r"C:\music\clip.wav")
YOUTUBE_URL: str | None = None  # e.g. "https://www.youtube.com/watch?v=..."

WORKDIR = DATA / "run"
WORKDIR.mkdir(parents=True, exist_ok=True)

SONG_WAV = WORKDIR / "song.wav"
STEMS_DIR = WORKDIR / "demucs_out"
OUT_GP5 = WORKDIR / "proof.gp5"

for exe in ("ffmpeg",):
    if shutil.which(exe) is None:
        raise RuntimeError(f"{exe} not found on PATH — install ffmpeg and retry.")

print("BACKEND_ROOT", BACKEND_ROOT)
print("WORKDIR", WORKDIR)

BACKEND_ROOT C:\Users\Irankunda\OneDrive\Desktop\Workspace\harmoniq\backend
WORKDIR C:\Users\Irankunda\OneDrive\Desktop\Workspace\harmoniq\backend\data\research_notebook\run


## 1) Ingest

- **Local file:** set `LOCAL_AUDIO` above.
- **YouTube (optional):** set `YOUTUBE_URL`; requires `yt-dlp` on PATH.

All paths are then normalized to **44.1 kHz mono** per `README.md`.

In [2]:
import sys

sys.path.insert(0, str(BACKEND_ROOT))

from app.pipeline_proof import (
    DEMUCS_MODEL,
    basic_pitch_predict_events,
    build_gp5_from_note_events,
    ffmpeg_normalize_wav,
    find_guitar_stem,
    librosa_summarize,
    run_demucs_htdemucs_6s,
    yt_dlp_download_wav,
)

raw_candidate: Path | None = None
if LOCAL_AUDIO is not None:
    raw_candidate = Path(LOCAL_AUDIO).expanduser().resolve()
    if not raw_candidate.is_file():
        raise FileNotFoundError(raw_candidate)
elif YOUTUBE_URL:
    dl_dir = WORKDIR / "yt_dl"
    dl_dir.mkdir(parents=True, exist_ok=True)
    raw_candidate = yt_dlp_download_wav(YOUTUBE_URL, dl_dir)
else:
    raise ValueError("Set LOCAL_AUDIO or YOUTUBE_URL in the config cell.")

ffmpeg_normalize_wav(raw_candidate, SONG_WAV)
print("Normalized song:", SONG_WAV, "bytes", SONG_WAV.stat().st_size)

ValueError: Set LOCAL_AUDIO or YOUTUBE_URL in the config cell.

## 2) Demucs (`htdemucs_6s`)

Produces six stems; the proof uses **`guitar.wav`**. Runtime: minutes on CPU for a full track, roughly tens of seconds to a few minutes on GPU depending on length.

In [ ]:
STEMS_DIR.mkdir(parents=True, exist_ok=True)
_ = run_demucs_htdemucs_6s(SONG_WAV, STEMS_DIR)
guitar_wav = find_guitar_stem(STEMS_DIR / DEMUCS_MODEL)
print("Guitar stem:", guitar_wav)

## 3) Librosa (tempo, beat times, key estimate)

Segmentation here is a stub (`full` span); later commits add structure detection.

In [ ]:
summary = librosa_summarize(guitar_wav)
print("duration_s", round(summary.duration_s, 3))
print("tempo_bpm", round(summary.tempo_bpm, 2))
print("key", summary.key_name)
print("beats (first 8)", [round(t, 3) for t in summary.beat_times_s[:8]])

## 4) Basic Pitch → note events

Adjust thresholds later in `app.pipeline_proof.basic_pitch_predict_events` if needed; for now we call the library default model with tempo aligned to Librosa for more consistent MIDI metadata.

In [ ]:
events = basic_pitch_predict_events(guitar_wav, midi_tempo=summary.tempo_bpm)
print("notes", len(events))
for e in events[:12]:
    print(f"  {e.start_s:.3f}-{e.end_s:.3f}s MIDI {e.pitch_midi}")

## 5) Export `.gp5`

Open in **Guitar Pro** or **AlphaTab** (desktop). The exporter quantizes to **quarter-note steps** and keeps a **monophonic** line per beat (highest simultaneous pitch).

In [ ]:
build_gp5_from_note_events(
    events,
    bpm=summary.tempo_bpm,
    output_gp5=OUT_GP5,
    title="Harmoniq research proof",
    artist="",
)
print("Wrote", OUT_GP5.resolve())

## CLI equivalents (for CI / automation later)

Same strings as `app.pipeline_proof.cli_equivalents_doc()`.

In [ ]:
from app.pipeline_proof import cli_equivalents_doc

print(cli_equivalents_doc())